# 🔷 Módulo 10 - Notebook 01: Indexación hexagonal H3

## 🗺️ Introducción a la Geolocalización Hexagonal

**Libro:** Saliendo de lo Pandito  
**Módulo:** 10 - Indexación Hexagonal Uber H3  
**Duración estimada:** 75 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Entender** qué es Uber H3 y por qué usar hexágonos  
✅ **Convertir** coordenadas lat/lon a índices H3  
✅ **Trabajar** con resoluciones jerárquicas (0-15)  
✅ **Agregar** datos espaciales por hexágonos  
✅ **Visualizar** mapas de densidad hexagonales

---

## 📋 Pre-requisitos

* ✅ Módulo 09 completado (GeoPandas)
* ✅ Conocimiento de coordenadas geográficas
* ✅ Familiaridad con agregaciones

---

## 📚 Contenido

1. ¿Qué es Uber H3?
2. Hexágonos vs Cuadrados
3. Resoluciones H3 (0-15)
4. Conversión lat/lon → H3
5. Agregación Espacial
6. Caso Integrador: Densidad de Ventas Hexagonal

---

## 💡 Por qué importa

**H3 revoluciona el análisis espacial:**

* 🔷 **Hexágonos:** Geometría más uniforme que cuadrados
* ⚡ **Performance:** Agregaciones 100x más rápidas
* 📊 **Jerarquía:** Zoom in/out sin pérdida de datos
* 🌍 **Global:** Sistema único mundial

**El estándar de facto para geoanalítica a escala**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (con coordenadas)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Extraer ubicaciones únicas de sucursales
    df_sucursales = df_ventas[['sucursal_id', 'sucursal_nombre', 'zona', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)
    
    # Agregar ventas totales por sucursal
    ventas_totales = df_ventas.groupby('sucursal_id')['ventas'].sum().reset_index()
    ventas_totales.columns = ['sucursal_id', 'ventas_totales']
    df_sucursales = df_sucursales.merge(ventas_totales, on='sucursal_id')
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros ventas: {len(df_ventas):,}")
    print(f"   🏪 Sucursales: {len(df_sucursales)}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n🗺️ Coordenadas disponibles:")
    print(f"   • lat (latitud): {df_sucursales['lat'].min():.4f} a {df_sucursales['lat'].max():.4f}")
    print(f"   • lon (longitud): {df_sucursales['lon'].min():.4f} a {df_sucursales['lon'].max():.4f}")
    
    print(f"\n🎯 Este notebook convertirá coordenadas REALES a índices H3")
    print(f"   Listo para indexación hexagonal")
    
    print(f"\n📊 Vista previa:")
    print(df_sucursales[['sucursal_nombre', 'zona', 'lat', 'lon', 'ventas_totales']].head())
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    df_sucursales = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Uber H3: Indexación Hexagonal del Planeta

### 🔷 ¿Qué es Uber H3?

**H3** es un sistema de indexación geoespacial jerárquico basado en **hexágonos** desarrollado por Uber.

**Concepto:**
* Divide el planeta en hexágonos de diferentes tamaños
* Cada hexágono tiene un **índice único** (ej: `8928308280fffff`)
* Sistema **jerárquico**: 16 resoluciones (0 a 15)

---

### 🔷 Hexágonos vs Cuadrados

**¿Por qué hexágonos y no cuadrados?**

| Característica | Hexágonos 🔷 | Cuadrados ⬛ |
|----------------|--------------|-------------|
| **Distancia al centro** | Uniforme | Variable (esquinas más lejos) |
| **Vecinos** | 6 equidistantes | 8 (4 lados + 4 esquinas a distinta distancia) |
| **Agregación** | Más precisa | Distorsión en bordes |
| **Visualización** | Natural para el ojo | Artificial |

**Ventaja clave:** En un hexágono, **todos los vecinos están a la misma distancia del centro**.

---

### 📊 Resoluciones H3 (0 a 15)

**Sistema jerárquico:** Cada nivel tiene hexágonos más pequeños.

| Resolución | Tamaño Hexágono | Uso Típico |
|------------|-----------------|------------|
| **0** | ~1,000 km² | Continentes |
| **3** | ~12 km² | Ciudades grandes |
| **5** | ~252 m² | Barrios |
| **7** | ~5.2 m² | Edificios |
| **9** | ~0.1 m² | Mesas/estantes |
| **12** | ~0.0009 m² | Centímetros |
| **15** | ~0.0000009 m² | Milímetros |

**Regla:**
* **Resolución baja** (0-5): Análisis regional/ciudad
* **Resolución media** (6-9): Análisis de barrio/calle
* **Resolución alta** (10-15): Indoor, IoT

---

### 🛠️ Conversión lat/lon → H3

**Sintaxis básica:**
```python
import h3

# Coordenadas: Mendoza Centro
lat = -32.8895
lon = -68.8458
resolucion = 9

# Convertir a H3
hex_id = h3.geo_to_h3(lat, lon, resolucion)
print(hex_id)  # Ej: '89a8100c54fffff'
```

**Inversa (H3 → lat/lon):**
```python
lat, lon = h3.h3_to_geo(hex_id)
```

---

### 📦 Agregación Espacial

**Flujo típico:**

```python
# 1. Convertir coordenadas a H3
df['h3_index'] = df.apply(
    lambda row: h3.geo_to_h3(row['lat'], row['lon'], 9),
    axis=1
)

# 2. Agregar por hexágono
hex_ventas = df.groupby('h3_index')['ventas'].sum().reset_index()

# 3. Visualizar en mapa
```

**Resultado:** Mapa de calor hexagonal con densidad de ventas.

---

### 🌍 Jerarquía H3

**Relación padre-hijo:**
```python
# Obtener hexágono padre (resolución menor)
parent = h3.h3_to_parent(hex_id, resolution=7)

# Obtener hexágonos hijos (resolución mayor)
children = h3.h3_to_children(hex_id, resolution=11)
```

**Uso:** Zoom in/out sin recalcular.

---

### 💼 Casos de Uso Empresariales

1. **Retail:** Densidad de ventas por zona hexagonal
2. **Logística:** Optimización de rutas y centros de distribución
3. **Inmobiliario:** Valoración por hexágono (precio/m²)
4. **Marketing:** Segmentación territorial precisa
5. **Seguros:** Pricing por riesgo geográfico hexagonal

---

### ⚡ Ventajas de H3

✅ **Performance:** Agregaciones hasta 100x más rápidas que lat/lon raw  
✅ **Precisión:** Geometría uniforme (sin distorsión de esquinas)  
✅ **Escalabilidad:** Del planeta entero a centímetros  
✅ **Interoperabilidad:** Estándar de facto (Uber, Google, AWS)

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🔷 UBER H3: INDEXACIÓN HEXAGONAL")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    import h3
    print(f"Versión de H3: {h3.__version__}")
except ImportError:
    print("⚠️  H3 no instalado. Ejecuta: %pip install h3")

print("\n🎯 En este notebook aprenderás:")
print("  • h3.geo_to_h3(lat, lon, resolution) - Convertir coordenadas")
print("  • h3.h3_to_geo(hex_id) - Obtener centro del hexágono")
print("  • Resoluciones H3 (0-15)")
print("  • Agregación espacial por hexágonos")

print("\n📖 Métodos clave:")
print("  - h3.geo_to_h3(lat, lon, res)  # lat/lon → H3")
print("  - h3.h3_to_geo(hex_id)  # H3 → lat/lon")
print("  - h3.h3_to_parent(hex_id, res)  # Padre")
print("  - h3.h3_to_children(hex_id, res)  # Hijos")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🔷 H3 con datos reales de Los Andes Market

### 📍 Índices H3 pre-calculados

El dataset `ventas_mensuales_mendoza_h3` ya incluye índices H3 a tres resoluciones:

| Columna | Resolución | Tamaño | Uso |
|---------|-----------|--------|-----|
| `h3_index` | 9 | ~174m | S Sucursal individual |
| `h3_res8` | 8 | ~461m | Agrupación de sucursales cercanas |
| `h3_res7` | 7 | ~1.22km | Zonas comerciales |

---

### 🛠️ Conversión manual con h3.geo_to_h3

```python
import h3

# Convertir lat/lon a índice H3 (resolución 9)
hex_id = h3.geo_to_h3(lat=-32.8895, lng=-68.8458, resolution=9)
# hex_id = '89c1e6402b7ffff' (ID único global)

# Obtener el centro del hexágono
lat_lon = h3.h3_to_geo(hex_id)
# (-32.8897, -68.8453) — centro del hexágono
```

---

### 💡 Preguntas de negocio
* ¿Cada sucursal está en un hexágono diferente o comparten?
* ¿Qué resolución es mejor para analizar Mendoza?
* ¿Cómo varía el número de hexágonos únicos entre res 7, 8 y 9?

In [0]:
import pandas as pd
import h3
import plotly.express as px

print("🔷 H3 CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_ventas is not None:
    print("\n1️⃣  ÍNDICES H3 PRE-CALCULADOS EN EL DATASET")
    print("-"*70)

    h3_cols = ['h3_index', 'h3_res8', 'h3_res7']
    for col in h3_cols:
        if col in df_ventas.columns:
            n_unique = df_ventas[col].nunique()
            print(f"\n   {col}: {n_unique} hexágonos únicos")

    print("\n   Ejemplo de índices H3 por sucursal:")
    cols_show = ['sucursal_nombre', 'lat', 'lon', 'h3_index', 'h3_res8', 'h3_res7']
    cols_available = [c for c in cols_show if c in df_ventas.columns]
    df_suc = df_ventas[cols_available].drop_duplicates()
    print(df_suc.head(10))

    print("\n" + "="*70)
    print("\n2️⃣  CONVERSIÓN MANUAL: lat/lon → H3")
    print("-"*70)

    # Convertir manualmente la primera sucursal
    primera = df_suc.iloc[0]
    lat, lon = primera['lat'], primera['lon']

    for res in [7, 8, 9]:
        hex_id = h3.geo_to_h3(lat=lat, lng=lon, res=res)
        centro = h3.h3_to_geo(hex_id)
        print(f"\n   Resolución {res}:")
        print(f"      lat/lon original: ({lat:.4f}, {lon:.4f})")
        print(f"      H3 ID: {hex_id}")
        print(f"      Centro hex: ({centro[0]:.4f}, {centro[1]:.4f})")

    print("\n   💡 El centro del hexágono difiere ligeramente del punto original")

    print("\n" + "="*70)
    print("\n3️⃣  COMPARACIÓN DE RESOLUCIONES")
    print("-"*70)

    for col, res_label in [("h3_res7", "Res 7 (~1.22km)"), ("h3_res8", "Res 8 (~461m)"), ("h3_index", "Res 9 (~174m)")]:
        if col in df_ventas.columns:
            n_hex = df_ventas[col].nunique()
            ventas_por_hex = df_ventas.groupby(col)['ventas'].sum()
            print(f"\n   {res_label} ({col}):")
            print(f"      Hexágonos únicos: {n_hex}")
            print(f"      Ventas promedio por hex: ${ventas_por_hex.mean():,.0f}")
            print(f"      Ventas máx en un hex: ${ventas_por_hex.max():,.0f}")

    print("\n" + "="*70)
    print("\n4️⃣  JERARQUÍA PADRE-HIJO")
    print("-"*70)

    # Tomar un H3 de resolución 9 y subir a res 7
    if 'h3_index' in df_ventas.columns:
        hex_res9 = df_ventas['h3_index'].iloc[0]
        padre_res7 = h3.h3_to_parent(hex_res9, 7)
        padre_res8 = h3.h3_to_parent(hex_res9, 8)

        print(f"\n   Hexágono res 9: {hex_res9}")
        print(f"   Padre res 8:    {padre_res8}")
        print(f"   Padre res 7:    {padre_res7}")

        # Cuántos hijos res 9 tiene el padre res 7
        hijos = h3.h3_to_children(padre_res7, 9)
        print(f"\n   Hijos res 9 del padre res 7: {len(list(hijos))} hexágonos")

    print("\n" + "="*70)
    print("\n5️⃣  VECINOS: k_ring (hexágonos adyacentes)")
    print("-"*70)

    if 'h3_res8' in df_ventas.columns:
        hex_res8 = df_ventas['h3_res8'].iloc[0]
        vecinos = h3.grid_disk(hex_res8, 1)
        vecinos_k2 = h3.grid_disk(hex_res8, 2)
        print(f"\n   Hexágono: {hex_res8}")
        print(f"   Vecinos k=1 (radio 1): {len(vecinos)} hexágonos (incluye el mismo)")
        print(f"   Vecinos k=2 (radio 2): {len(vecinos_k2)} hexágonos")
        print("\n   💡 k_ring incluye el hexágono central + anillos concéntricos")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# 🎯 OPCIONAL: Usar datos con H3 precalculado desde Unity Catalog

# Descomentar para usar datos reales:
"""
import pandas as pd
import h3

print("💾 Cargando datos con índices H3 desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    print(f"✅ Datos cargados: {len(df_ventas):,} registros")
    print(f"   Sucursales: {df_ventas['sucursal_id'].nunique()}")
    
    print(f"\n🕸️ Índices H3 Disponibles:")
    print("   • h3_index (res 9): ~174m por hexágono")
    print("   • h3_res8 (res 8):  ~461m por hexágono")
    print("   • h3_res7 (res 7):  ~1.22km por hexágono")
    
    # Obtener sucursales únicas con sus H3
    df_sucursales = df_ventas[[
        'sucursal_id', 'sucursal_nombre', 'lat', 'lon', 
        'h3_index', 'h3_res8', 'h3_res7', 'zona'
    ]].drop_duplicates()
    
    print(f"\n📊 Ejemplo de índices H3:")
    display(df_sucursales[['sucursal_id', 'h3_index', 'h3_res8', 'h3_res7']].head())
    
    print(f"\n💡 Operaciones H3 disponibles:")
    print("\n1️⃣ Obtener vecinos de un hexágono:")
    print("   h3_id = df_sucursales.iloc[0]['h3_index']")
    print("   vecinos = h3.grid_disk(h3_id, k=1)  # k=1: vecinos inmediatos")
    
    print("\n2️⃣ Convertir entre resoluciones:")
    print("   h3_parent = h3.cell_to_parent(h3_id, res=7)  # Subir a res 7")
    print("   h3_children = h3.cell_to_children(h3_id, res=10)  # Bajar a res 10")
    
    print("\n3️⃣ Obtener polígono del hexágono:")
    print("   boundary = h3.cell_to_boundary(h3_id)  # Lista de (lat, lon)")
    
    print("\n4️⃣ Calcular distancia entre hexágonos:")
    print("   distance = h3.grid_distance(h3_id1, h3_id2)  # En saltos de hex")
    
    print("\n5️⃣ Agregaciones espaciales:")
    print("   # Ventas por hexágono (resolución 8)")
    print("   ventas_h3 = df_ventas.groupby('h3_res8')['ventas'].sum()")
    
    # Ejemplo práctico: Calcular densidad de sucursales
    print(f"\n🎯 Ejemplo: Densidad de Sucursales por Hexágono (res 7)")
    densidad = df_sucursales.groupby('h3_res7')['sucursal_id'].count()
    print(f"   Total hexágonos ocupados: {len(densidad)}")
    print(f"   Sucursales por hexágono:")
    for h3_id, count in densidad.items():
        print(f"      {h3_id}: {count} sucursales")
    
    print(f"\n💡 Variables disponibles:")
    print("   • df_ventas: DataFrame completo con ventas e índices H3")
    print("   • df_sucursales: Sucursales únicas con sus H3")
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
# Demostración del concepto de Indexación H3
latitud = -34.6037
longitud = -58.3816
resolucion = 8

print(f"Coordenadas de Sucursal: Lat {latitud}, Lon {longitud}")
print(f"Resolución H3 seleccionada: {resolucion}")
print("Indexación espacial hexagonal lista para agregación de densidad comercial.")



## 🎓 Conclusiones del notebook 10_01

### ✅ Lo que aprendiste

1. **Uber H3 — indexación hexagonal global:**
   - `h3.geo_to_h3(lat, lon, resolution)` convierte coordenadas a índice H3
   - Cada hexágono tiene un ID único (ej: `8928308280fffff`)
   - Sistema jerárquico de 16 resoluciones (0 a 15)

2. **Hexágonos vs Cuadrados:**
   - 6 vecinos equidistantes (vs 8 a distancias variables en cuadrados)
   - Sin distorsión en bordes ni esquinas
   - Agregación espacial más precisa y uniforme

3. **Resoluciones H3 (0-15):**
   - Res 0: ~1,000 km² (continentes) → Res 15: ~0.0000009 m² (milímetros)
   - Baja (0-5): región/ciudad → Media (6-9): barrio/calle → Alta (10-15): indoor/IoT
   - `h3.h3_to_parent()` sube, `h3.h3_to_children()` baja de resolución

4. **Conversión bidireccional:**
   - `h3.geo_to_h3(lat, lon, res)` — coordenadas → H3
   - `h3.h3_to_geo(hex_id)` — H3 → centro del hexágono (lat, lon)
   - `h3.cell_to_boundary(hex_id)` — H3 → vértices del polígono

5. **Agregación espacial con H3:**
   - `df.groupby('h3_index')['ventas'].sum()` — agregación por hexágono
   - Mapas de densidad hexagonal (heatmap espacial)
   - Performance 100x superior a operaciones con lat/lon raw

---

### 🎯 Reglas de Oro

👉 **Regla #1: Elegir resolución según escala del análisis**
```python
# MALO: res 9 (~0.1 m²) para analizar ciudades enteras
hex_id = h3.geo_to_h3(lat, lon, 9)

# BUENO: res 7 (~5.2 km²) para análisis urbano
df['h3_index'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 7), axis=1)
ventas_hex = df.groupby('h3_index')['ventas'].sum()
```

👉 **Regla #2: Para comparar hexágonos, usar la MISMA resolución**
```python
# MALO: mezclar resoluciones (comparar peras con manzanas)
df['h3_res7'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 7), axis=1)
df['h3_res9'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 9), axis=1)

# BUENO: fijar una resolución y agregar todo bajo esa misma grilla
df['h3_index'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 8), axis=1)
ventas_por_hex = df.groupby('h3_index')['ventas'].sum().reset_index()
```

👉 **Regla #3: geo_to_h3 recibe (lat, lon), NO (lon, lat)**
```python
# MALO: orden incorrecto (lon, lat)
hex_id = h3.geo_to_h3(-68.8458, -32.8895, 8)  # hexágono erróneo

# BUENO: latitud primero, longitud después
hex_id = h3.geo_to_h3(-32.8895, -68.8458, 8)  # Mendoza Centro correcto
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Coordenadas → índice H3 | `h3.geo_to_h3(lat, lon, res)` |
| Índice H3 → centro (lat, lon) | `h3.h3_to_geo(hex_id)` |
| Índice H3 → polígono | `h3.cell_to_boundary(hex_id)` |
| Subir de resolución (zoom out) | `h3.h3_to_parent(hex_id, res_menor)` |
| Bajar de resolución (zoom in) | `h3.h3_to_children(hex_id, res_mayor)` |
| Vecinos de un hexágono | `h3.grid_disk(hex_id, k=1)` |
| Distancia entre hexágonos | `h3.grid_distance(hex_id1, hex_id2)` |
| Agregar ventas por hexágono | `df.groupby('h3_index')['ventas'].sum()` |
| Análisis urbano (Mendoza) | Resolución 7-8 (~1-5 km²) |
| Análisis de barrio | Resolución 8-9 (~100-500 m²) |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔷 ¡Indexación hexagonal Uber H3 dominada!</h3>
  <p><i>"H3 convierte coordenadas caóticas en grillas uniformes: el estándar de facto para geoanalítica a escala."</i></p>
</div>